# Summary Meta Data

In [13]:
import pandas as pd
from IPython.display import display

from irp.data import companies, fundamentals, prices

In [14]:
comp = companies()
print(f'Total tickers: {len(comp)}')
print(f'Markets: {comp["Market"].value_counts().to_dict()}')
print(f'Sectors: {comp["Sector"].nunique()}  |  Industries: {comp["Industry"].nunique()}')

Total tickers: 6612
Markets: {'us': 6580, 'de': 32}
Sectors: 12  |  Industries: 74


## Coverage by Market

In [15]:
display(
    comp.groupby('Market')
    .agg(tickers=('Ticker', 'count'), sectors=('Sector', 'nunique'), industries=('Industry', 'nunique'))
    .reset_index()
)

,Market,tickers,sectors,industries
0,de,32,11,24
1,us,6542,12,74


## Coverage by Sector

In [16]:
display(
    comp.groupby(['Market', 'Sector'])
    .agg(tickers=('Ticker', 'count'), industries=('Industry', 'nunique'))
    .reset_index()
    .sort_values(['Market', 'tickers'], ascending=[True, False])
    .reset_index(drop=True)
)

,Market,Sector,tickers,industries
0,de,Consumer Cyclical,6,3
1,de,Financial Services,5,5
2,de,Healthcare,4,4
3,de,Basic Materials,3,2
4,de,Industrials,3,3
5,de,Business Services,2,1
6,de,Consumer Defensive,2,1
7,de,Real Estate,2,1
8,de,Technology,2,2
9,de,Utilities,2,1


## Missing Fields

In [17]:
_check_cols = ['Ticker', 'ISIN', 'CIK', 'Sector', 'Industry', 'Number Employees', 'Main Currency', 'End of financial year (month)']
for _col in _check_cols:
    _n = comp[_col].isna().sum()
    print(f'{_col:<40s}  {_n:>5} missing  ({100 * _n / len(comp):.1f}%)')

Ticker                                       38 missing  (0.6%)
ISIN                                       1201 missing  (18.2%)
CIK                                          42 missing  (0.6%)
Sector                                      311 missing  (4.7%)
Industry                                    311 missing  (4.7%)
Number Employees                            833 missing  (12.6%)
Main Currency                                 0 missing  (0.0%)
End of financial year (month)                35 missing  (0.5%)


## Duplicate Tickers

In [18]:
_dups = comp[comp.duplicated('Ticker', keep=False)].sort_values('Ticker')
print(f'Duplicate tickers: {_dups["Ticker"].nunique()}')
if len(_dups):
    display(_dups[['Ticker', 'Company Name', 'Market', 'SrcId']])
else:
    print('None.')

Duplicate tickers: 0


,Ticker,Company Name,Market,SrcId
6574,NaN,NaN,us,20095034
6575,NaN,NaN,us,20215308
6576,NaN,NaN,us,18692750
6577,NaN,NaN,us,18847915
6578,NaN,NaN,us,18538670
6579,NaN,NaN,us,18657366
6580,NaN,NaN,us,18667300
6581,NaN,NaN,us,14159407
6582,NaN,NaN,us,14159427
6583,NaN,NaN,us,15112475


## Tickers Without Fundamentals

In [19]:
_fund_tickers = (
    set(fundamentals(statement='income')['Ticker'])
    | set(fundamentals(statement='balance')['Ticker'])
    | set(fundamentals(statement='cashflow')['Ticker'])
)
_no_fund = sorted(set(comp['Ticker'].dropna()) - _fund_tickers)
print(f'Tickers with no fundamentals: {len(_no_fund)}')
print(_no_fund)

Tickers with no fundamentals: 2127
['AACB', 'AAC_delisted', 'AAME', 'AATP', 'AB', 'ABAX', 'ABCB', 'ABCD', 'ABEV', 'ABTI', 'ABTX', 'ABX', 'ACAT', 'ACBI', 'ACBM', 'ACDC', 'ACET_delisted', 'ACFN', 'ACGL', 'ACIC', 'ACIU', 'ACNB', 'ACPW', 'ACRHF', 'ACT', 'ACTU', 'ACW', 'ADEA', 'ADPT_delisted', 'ADSE', 'ADTM', 'AEL', 'AEM', 'AEPI', 'AER', 'AES', 'AESI', 'AET', 'AEXA', 'AFAR', 'AFBI', 'AFG', 'AFL', 'AFRI', 'AFSI', 'AFYA', 'AG', 'AGBA', 'AGHI', 'AGI', 'AGN', 'AGN_old', 'AGO', 'AGRO', 'AGSS', 'AHFD', 'AHL', 'AHMA', 'AIB', 'AIFE', 'AIG', 'AII', 'AIXI', 'AIZ', 'AJBI', 'AJIA', 'AJX', 'AKBA', 'AKO-B', 'AKRX', 'AKS', 'ALCY', 'ALDF', 'ALID', 'ALJ', 'ALK', 'ALL', 'ALLY', 'ALNT', 'ALPC', 'ALR', 'ALRS', 'ALTA', 'ALTMS', 'ALTR_delisted', 'ALV.DE', 'ALVO', 'AMAL', 'AMBC', 'AMBI', 'AMBP', 'AMEH', 'AMMJ', 'AMNB', 'AMRU', 'AMSF', 'AMTB', 'AMTD', 'AMV', 'AMX', 'ANDV', 'ANDX', 'ANTA', 'AOGO', 'APC', 'APG', 'APIC', 'APLM', 'APTI', 'APTO', 'APU', 'ARA', 'ARBB', 'ARBE', 'ARCW', 'AREX', 'ARG', 'ARGC', 'ARGO', 'ARG

## Tickers Without Prices

In [20]:
_price_tickers = set(prices()['Ticker'])
_no_prices = sorted(set(comp['Ticker'].dropna()) - _price_tickers)
print(f'Tickers with no prices: {len(_no_prices)}')
print(_no_prices)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tickers with no prices: 187
['A21', 'AAC_delisted', 'ACPW', 'AFGS', 'AH', 'AHFC', 'AINV', 'AKU', 'ALTMS', 'ALTR_delisted', 'ANGN', 'APEN', 'ARRY_delisted', 'ARTEC', 'ATAX', 'ATHN_delisted', 'AUDH', 'BCEI_delisted', 'BCLR', 'BCSI', 'BEXP', 'BLUD', 'BOMN', 'BV_old', 'C649', 'CAPC', 'CA_delisted', 'CBTX', 'CCFI', 'CECE', 'CERBE', 'CHAA', 'CHDX', 'CICC', 'CIK0001446', 'CIK0001536', 'CIK1479320', 'CINR', 'CK0000081061', 'CK00007861', 'CK00011568', 'CK00015317', 'CK00015503', 'CK0001584423', 'CK00015847', 'CLSN', 'CMBG', 'CMLF', 'COOP', 'COR_delisted', 'CSWG', 'CTA-PB', 'CTG_delisted', 'CTOP', 'CXBS', 'CYNX', 'DDMG', 'DGTC', 'DHRM', 'DMDW', 'DUNE', 'DYNS', 'DYN_delisted', 'EFCT', 'ELLI', 'ENOB', 'ESC', 'EVHC', 'EVK', 'EWRX', 'EXAM', 'FACT-UN', 'FEC', 'FFKY', 'FLWD', 'FWV', 'GASE', 'GBL', 'GGAA', 'GLBL_delisted', 'GLOG-PA', 'GNLK', 'GNZR', 'GOBK', 'GPCM', 'HBKS', 'HGSH', 'HOMR', 'HUSI', 'ICR-PA', 'IDRA', 'IIVI', 'IKNX', 'IMMC', 'INFOR', 'INTEQ', 'ISSM', 'ITNM', 'IVC', 'JATT-UN', 'JCG', 'JDAS'